# 5.3. Cài đặt minh hoạ mô hình con trỏ lặp (Iterator Pattern)

**Mục tiêu**

1. Dùng Python cài đặt một bộ thực thi truy vấn thu nhỏ theo mô hình iterator: mỗi toán tử vật lý là một lớp có 3 phương thức `open()`, `get_next()`, `close()`.
2. Chạy 3 thí nghiệm để **nhìn thấy** cách dữ liệu chảy từng dòng qua các toán tử (xử lý luồng).
3. Với mỗi thí nghiệm, chạy câu SQL tương ứng bằng `EXPLAIN ANALYZE` trên PostgreSQL để **chứng minh** PostgreSQL áp dụng đúng nguyên lý đó.

| Thí nghiệm | Nội dung | Câu SQL đối chiếu |
|---|---|---|
| 1 | Vết lời gọi `open / get_next / close` | `SELECT lname FROM employee WHERE dno = 5` |
| 2 | `LIMIT` làm các toán tử dưới dừng sớm | thêm `LIMIT 2` |
| 3 | Toán tử chặn `Sort` làm đứt luồng | thêm `ORDER BY salary DESC LIMIT 2` |

Phần Python dùng 8 nhân viên của sách để vết ngắn, dễ đọc. Phần PostgreSQL chạy trên CSDL `company` khoảng 100.000 dòng.
Notebook chỉ dùng Python chuẩn, không cần cài thêm thư viện.

## 1. Dữ liệu: bảng EMPLOYEE của sách

Chỉ lấy các cột cần dùng. Thứ tự các dòng giống thứ tự chèn trong script của sách.

In [1]:
EMPLOYEE = [
    {"fname": "John",     "lname": "Smith",   "ssn": "123456789", "salary": 30000, "dno": 5},
    {"fname": "Franklin", "lname": "Wong",    "ssn": "333445555", "salary": 40000, "dno": 5},
    {"fname": "Alicia",   "lname": "Zelaya",  "ssn": "999887777", "salary": 25000, "dno": 4},
    {"fname": "Jennifer", "lname": "Wallace", "ssn": "987654321", "salary": 43000, "dno": 4},
    {"fname": "Ramesh",   "lname": "Narayan", "ssn": "666884444", "salary": 38000, "dno": 5},
    {"fname": "Joyce",    "lname": "English", "ssn": "453453453", "salary": 25000, "dno": 5},
    {"fname": "Ahmad",    "lname": "Jabbar",  "ssn": "987987987", "salary": 25000, "dno": 4},
    {"fname": "James",    "lname": "Borg",    "ssn": "888665555", "salary": 55000, "dno": 1},
]
print(len(EMPLOYEE), "nhân viên")

8 nhân viên


## 2. Lớp cơ sở `Iterator`

Mọi toán tử kế thừa lớp này. Lớp cơ sở lo phần **in vết** (thụt lề theo tầng của cây), còn mỗi toán tử chỉ cài đặt `_open()`, `_next()`, `_close()`.

| Phương thức | Việc làm (theo lý thuyết 5.3) |
|---|---|
| `open()` | Mở các toán tử con, cấp phát bộ đệm, khởi tạo trạng thái |
| `get_next()` | Trả về **đúng 1 dòng**, hoặc `None` khi hết dữ liệu |
| `close()` | Đóng các toán tử con, thu hồi tài nguyên |

In [2]:
TRACE = {"on": True, "depth": 0}

def log(msg):
    if TRACE["on"]:
        print("    " * TRACE["depth"] + msg)


class Iterator:
    def __init__(self, name, child=None):
        self.name = name
        self.child = child

    def open(self):
        log(f"{self.name}.open()")
        TRACE["depth"] += 1
        if self.child:
            self.child.open()
        self._open()
        TRACE["depth"] -= 1

    def get_next(self):
        log(f"{self.name}.get_next()")
        TRACE["depth"] += 1
        row = self._next()
        TRACE["depth"] -= 1
        log(f"<- {self.name} trả lên: {row['lname'] if row else 'None (hết dữ liệu)'}")
        return row

    def close(self):
        log(f"{self.name}.close()")
        TRACE["depth"] += 1
        self._close()
        if self.child:
            self.child.close()
        TRACE["depth"] -= 1

    def _open(self):  pass
    def _next(self):  raise NotImplementedError
    def _close(self): pass

## 3. Các toán tử vật lý

| Lớp Python | Nút tương ứng trong `EXPLAIN` của PostgreSQL | Loại |
|---|---|---|
| `TableScan` | `Seq Scan` | Luồng |
| `Select` | `Filter` (nằm trong nút Seq Scan) | Luồng |
| `Project` | danh sách cột trong SELECT | Luồng |
| `Limit` | `Limit` | Luồng, dừng sớm |
| `Sort` | `Sort` | **Chặn**: phải đọc hết đầu vào mới trả được dòng đầu |

In [3]:
class TableScan(Iterator):
    # Đọc tuần tự từng dòng của bảng
    def __init__(self, table, name="employee"):
        super().__init__(f"TableScan({name})")
        self.table = table

    def _open(self):
        self.pos = 0
        self.rows_read = 0            # đếm số dòng đã đọc

    def _next(self):
        if self.pos >= len(self.table):
            return None
        row = self.table[self.pos]
        self.pos += 1
        self.rows_read += 1
        log(f"[đọc dòng {self.rows_read}: {row['lname']}, dno={row['dno']}]")
        return row


class Select(Iterator):
    # Phép chọn: chỉ trả lên dòng thoả điều kiện
    def __init__(self, child, predicate, desc):
        super().__init__(f"Select({desc})", child)
        self.pred = predicate

    def _next(self):
        while True:
            row = self.child.get_next()
            if row is None:
                return None
            if self.pred(row):
                return row            # trả NGAY, không chờ đọc hết bảng
            log(f"[loại {row['lname']}: không thoả điều kiện]")


class Project(Iterator):
    # Phép chiếu: giữ lại một số cột
    def __init__(self, child, cols):
        super().__init__(f"Project({', '.join(cols)})", child)
        self.cols = cols

    def _next(self):
        row = self.child.get_next()
        return None if row is None else {c: row[c] for c in self.cols}


class Limit(Iterator):
    # Ngừng gọi toán tử con khi đã đủ n dòng
    def __init__(self, child, n):
        super().__init__(f"Limit({n})", child)
        self.n = n

    def _open(self):
        self.count = 0

    def _next(self):
        if self.count >= self.n:
            log("[đã đủ dòng -> không gọi xuống nữa]")
            return None
        row = self.child.get_next()
        if row is not None:
            self.count += 1
        return row


class Sort(Iterator):
    # Toán tử CHẶN: open() đọc hết đầu vào và sắp xếp, get_next() mới trả từng dòng
    def __init__(self, child, key, desc=False):
        super().__init__(f"Sort({key}{' DESC' if desc else ''})", child)
        self.key, self.desc = key, desc

    def _open(self):
        self.buffer = []              # cấp phát bộ đệm
        while (row := self.child.get_next()) is not None:
            self.buffer.append(row)
        self.buffer.sort(key=lambda r: r[self.key], reverse=self.desc)
        self.i = 0
        log(f"[đã đọc hết và sắp xếp {len(self.buffer)} dòng]")

    def _next(self):
        if self.i >= len(self.buffer):
            return None
        self.i += 1
        return self.buffer[self.i - 1]

    def _close(self):
        self.buffer = None            # thu hồi bộ đệm

## 4. Bộ thực thi

Đúng như bộ thực thi của hệ quản trị CSDL: gọi `open()` ở gốc cây, lặp `get_next()` cho tới khi nhận `None`, rồi `close()`.

In [4]:
def run(plan, trace=True):
    TRACE.update(on=trace, depth=0)
    plan.open()
    result = []
    while (row := plan.get_next()) is not None:
        result.append(row)
        log(f"===> KẾT QUẢ dòng {len(result)}: {row}")
    plan.close()
    return result

## 5. Thí nghiệm 1: Vết lời gọi `open / get_next / close`

**Câu truy vấn:** `SELECT lname FROM employee WHERE dno = 5`

**Cây toán tử:** `Project(lname) → Select(dno = 5) → TableScan(employee)`

**Cần quan sát trong vết:**
- `open()` được gọi lan từ gốc xuống lá.
- Mỗi `get_next()` ở gốc kéo theo chuỗi `get_next()` xuống tận `TableScan`, và **chỉ một dòng** được đọc lên mỗi lần.
- Dòng thoả điều kiện đi thẳng lên tới gốc thành kết quả (`===> KẾT QUẢ`) **trước khi** dòng tiếp theo được đọc. Đó là xử lý luồng: không có bảng kết quả trung gian.
- Dòng không thoả (Zelaya, Wallace...) bị `Select` loại và gọi tiếp xuống dưới.
- Khi `TableScan` hết dữ liệu, `None` được truyền lên, rồi `close()` lan từ gốc xuống.

In [5]:
scan = TableScan(EMPLOYEE)
plan = Project(Select(scan, lambda r: r["dno"] == 5, "dno = 5"), ["lname"])

result = run(plan)
print(f"\nKết quả: {result}")
print(f"TableScan đã đọc {scan.rows_read}/{len(EMPLOYEE)} dòng")

Project(lname).open()
    Select(dno = 5).open()
        TableScan(employee).open()
Project(lname).get_next()
    Select(dno = 5).get_next()
        TableScan(employee).get_next()
            [đọc dòng 1: Smith, dno=5]
        <- TableScan(employee) trả lên: Smith
    <- Select(dno = 5) trả lên: Smith
<- Project(lname) trả lên: Smith
===> KẾT QUẢ dòng 1: {'lname': 'Smith'}
Project(lname).get_next()
    Select(dno = 5).get_next()
        TableScan(employee).get_next()
            [đọc dòng 2: Wong, dno=5]
        <- TableScan(employee) trả lên: Wong
    <- Select(dno = 5) trả lên: Wong
<- Project(lname) trả lên: Wong
===> KẾT QUẢ dòng 2: {'lname': 'Wong'}
Project(lname).get_next()
    Select(dno = 5).get_next()
        TableScan(employee).get_next()
            [đọc dòng 3: Zelaya, dno=4]
        <- TableScan(employee) trả lên: Zelaya
        [loại Zelaya: không thoả điều kiện]
        TableScan(employee).get_next()
            [đọc dòng 4: Wallace, dno=4]
        <- TableScan(employee)

### Đối chiếu trên PostgreSQL (chạy trong pgAdmin)

```sql
EXPLAIN ANALYZE SELECT lname FROM employee WHERE dno = 5;
```

**Cần chỉ ra trong kết quả:**
- Kế hoạch là một cây nút, mỗi nút là một iterator; ở đây là nút `Seq Scan on employee` có dòng `Filter: (dno = 5)`. PostgreSQL gộp `TableScan` và `Select` vào chung một nút, nhưng nguyên lý giống hệt: đọc từng dòng, kiểm tra điều kiện, trả ngay dòng thoả.
- `actual time=A..B`: A rất nhỏ (gần 0 ms) nghĩa là **dòng kết quả đầu tiên được trả ra gần như ngay lập tức**, không chờ đọc hết bảng. Đây là bằng chứng của xử lý luồng.
- `Rows Removed by Filter`: số dòng bị loại, tương ứng các dòng `[loại ...]` trong vết Python.

> *(Dán ảnh kết quả EXPLAIN ANALYZE vào đây khi viết báo cáo)*

## 6. Thí nghiệm 2: `LIMIT` làm các toán tử dưới dừng sớm

**Câu truy vấn:** `SELECT lname FROM employee WHERE dno = 5 LIMIT 2`

**Cây toán tử:** `Limit(2) → Project(lname) → Select(dno = 5) → TableScan(employee)`

**Cần quan sát:** sau khi có đủ 2 dòng, `Limit` trả `None` mà **không gọi `get_next()` xuống nữa**, nên `TableScan` dừng sớm, không đọc hết bảng.

In [6]:
scan = TableScan(EMPLOYEE)
plan = Limit(Project(Select(scan, lambda r: r["dno"] == 5, "dno = 5"), ["lname"]), 2)

result = run(plan)
print(f"\nKết quả: {result}")
print(f"TableScan chỉ đọc {scan.rows_read}/{len(EMPLOYEE)} dòng")

Limit(2).open()
    Project(lname).open()
        Select(dno = 5).open()
            TableScan(employee).open()
Limit(2).get_next()
    Project(lname).get_next()
        Select(dno = 5).get_next()
            TableScan(employee).get_next()
                [đọc dòng 1: Smith, dno=5]
            <- TableScan(employee) trả lên: Smith
        <- Select(dno = 5) trả lên: Smith
    <- Project(lname) trả lên: Smith
<- Limit(2) trả lên: Smith
===> KẾT QUẢ dòng 1: {'lname': 'Smith'}
Limit(2).get_next()
    Project(lname).get_next()
        Select(dno = 5).get_next()
            TableScan(employee).get_next()
                [đọc dòng 2: Wong, dno=5]
            <- TableScan(employee) trả lên: Wong
        <- Select(dno = 5) trả lên: Wong
    <- Project(lname) trả lên: Wong
<- Limit(2) trả lên: Wong
===> KẾT QUẢ dòng 2: {'lname': 'Wong'}
Limit(2).get_next()
    [đã đủ dòng -> không gọi xuống nữa]
<- Limit(2) trả lên: None (hết dữ liệu)
Limit(2).close()
    Project(lname).close()
        Select(d

### Đối chiếu trên PostgreSQL

```sql
EXPLAIN ANALYZE SELECT lname FROM employee WHERE dno = 5 LIMIT 2;
```

**Cần chỉ ra trong kết quả:**
- Nút `Limit` nằm trên nút `Seq Scan`, giống cây Python.
- Ở nút `Seq Scan`: `rows=2` (thực tế) và `Rows Removed by Filter` chỉ vài dòng, dù bảng có khoảng 100.008 dòng. Tức là PostgreSQL cũng **dừng đọc bảng** ngay khi `Limit` đủ 2 dòng.
- So sánh `Execution Time` với Thí nghiệm 1: nhỏ hơn rất nhiều.

> *(Dán ảnh kết quả vào đây)*

## 7. Thí nghiệm 3: Toán tử chặn `Sort` làm đứt luồng

**Câu truy vấn:** `SELECT lname, salary FROM employee WHERE dno = 5 ORDER BY salary DESC LIMIT 2`

**Cây toán tử:** `Limit(2) → Project(lname, salary) → Sort(salary DESC) → Select(dno = 5) → TableScan(employee)`

**Cần quan sát:** vẫn chỉ lấy 2 dòng như Thí nghiệm 2, nhưng `Sort` muốn biết ai lương cao nhất thì phải **đọc hết** đầu vào trong `open()` rồi mới trả được dòng đầu tiên. `Limit` không còn giúp dừng sớm được nữa.

In [7]:
scan = TableScan(EMPLOYEE)
plan = Limit(
    Project(
        Sort(Select(scan, lambda r: r["dno"] == 5, "dno = 5"), "salary", desc=True),
        ["lname", "salary"]),
    2)

result = run(plan)
print(f"\nKết quả: {result}")
print(f"TableScan phải đọc {scan.rows_read}/{len(EMPLOYEE)} dòng dù chỉ lấy 2 dòng")

Limit(2).open()
    Project(lname, salary).open()
        Sort(salary DESC).open()
            Select(dno = 5).open()
                TableScan(employee).open()
            Select(dno = 5).get_next()
                TableScan(employee).get_next()
                    [đọc dòng 1: Smith, dno=5]
                <- TableScan(employee) trả lên: Smith
            <- Select(dno = 5) trả lên: Smith
            Select(dno = 5).get_next()
                TableScan(employee).get_next()
                    [đọc dòng 2: Wong, dno=5]
                <- TableScan(employee) trả lên: Wong
            <- Select(dno = 5) trả lên: Wong
            Select(dno = 5).get_next()
                TableScan(employee).get_next()
                    [đọc dòng 3: Zelaya, dno=4]
                <- TableScan(employee) trả lên: Zelaya
                [loại Zelaya: không thoả điều kiện]
                TableScan(employee).get_next()
                    [đọc dòng 4: Wallace, dno=4]
                <- TableScan(employee) 

### Đối chiếu trên PostgreSQL

```sql
EXPLAIN ANALYZE
SELECT lname, salary FROM employee WHERE dno = 5
ORDER BY salary DESC LIMIT 2;
```

**Cần chỉ ra trong kết quả:**
- Cây `Limit → Sort → Seq Scan`, giống cây Python.
- Ở nút `Seq Scan`: `rows` thực tế khoảng 40.000 (toàn bộ nhân viên phòng 5), không còn là 2 như Thí nghiệm 2. Bảng bị đọc **hết**.
- Ở nút `Sort`: `actual time=A..B` với A gần bằng B, tức là dòng đầu tiên chỉ ra sau khi đã sắp xếp xong toàn bộ.
- `Sort Method: top-N heapsort`: PostgreSQL tối ưu thêm bằng cách chỉ giữ 2 dòng lớn nhất trong bộ nhớ, nhưng vẫn phải xem qua mọi dòng.
- `Execution Time` lớn hơn hẳn Thí nghiệm 2.

> *(Dán ảnh kết quả vào đây)*

## 8. Tổng hợp

| | Thí nghiệm 1 | Thí nghiệm 2 | Thí nghiệm 3 |
|---|---|---|---|
| Câu truy vấn | `WHERE dno = 5` | `+ LIMIT 2` | `+ ORDER BY salary DESC LIMIT 2` |
| Python: số dòng TableScan đọc (trên 8) | 8 | 2 | 8 |
| PostgreSQL: `rows` thực tế của Seq Scan | *(điền)* | *(điền)* | *(điền)* |
| PostgreSQL: Execution Time | *(điền)* | *(điền)* | *(điền)* |
| Kết luận | Dữ liệu chảy từng dòng | Luồng cho phép dừng sớm | Toán tử chặn làm đứt luồng |

### PostgreSQL cài đặt đúng mô hình này

Bộ thực thi của PostgreSQL (viết bằng C) tổ chức mỗi nút trong kết quả `EXPLAIN` thành một iterator, gọi là **mô hình Volcano**:

| Lý thuyết 5.3 | Notebook này | Hàm trong mã nguồn PostgreSQL |
|---|---|---|
| `Open()` | `open()` | `ExecInitNode()` |
| `Get_Next()` | `get_next()` | `ExecProcNode()` |
| `Close()` | `close()` | `ExecEndNode()` |

## 9. Kết luận

1. Mô hình iterator chuẩn hoá mọi toán tử vật lý về cùng giao diện `open / get_next / close`, nhờ đó các toán tử lắp ghép được thành cây kế hoạch bất kỳ.
2. Vì mỗi `get_next()` chỉ chuyển **một dòng** từ toán tử con lên toán tử cha, kết quả chảy liên tục từ lá lên gốc mà không cần lưu bảng trung gian. Đây chính là xử lý luồng (pipelining), giúp trả dòng đầu tiên rất sớm.
3. Kết hợp với `LIMIT`, xử lý luồng cho phép dừng đọc dữ liệu sớm (Thí nghiệm 2).
4. Toán tử chặn như `Sort` phải đọc hết đầu vào mới trả được dòng đầu tiên, nên làm đứt luồng (Thí nghiệm 3).
5. Kết quả `EXPLAIN ANALYZE` của PostgreSQL cho thấy đúng các hành vi trên, chứng minh hệ quản trị thực tế áp dụng mô hình iterator và xử lý luồng như lý thuyết.